[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-10-comet-install.ipynb)


<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 10 (*Overview of MT Quality Estimation*). A head start on **A10**.

# Install COMET and score one example

**Homework. About 25 minutes.**

COMET is a learned metric: a model that scores a translation, with or without a reference, and
agrees with human judgment better than BLEU or chrF does. Lecture 11 runs it on your A6
sentences; here you get it running, and write the one piece A11 reuses.

By the end you will be able to:

1. install COMET and score translations with a reference, and without one;
2. turn aligned sentences into the records COMET scores, the shape your A6 sentences go into;
3. say which of two translations each model prefers, against your own prediction.

You write one function, marked TODO; checks tell you when it is right, and the notebook runs
the rest. **Running in Google Colab?** Run all; it stops at the TODO until you write it. No GPU
is needed.


In [ ]:
# Install COMET in Colab. It takes a minute or two, and pip's "restart the session" notice can
# be ignored for this notebook.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet unbabel-comet


## The example

Two source sentences, two systems' translations of them, and a reference translation of each.
Line *n* of every list belongs to source sentence *n*: that alignment is what COMET relies on.


In [ ]:
sources = ["10 到 15 分钟可以送到吗", "Pode ser entregue dentro de 10 a 15 minutos?"]
system_1 = ["Can I receive my food in 10 to 15 minutes?", "Can it be delivered in 10 to 15 minutes?"]
system_2 = ["Can it be delivered within 10 to 15 minutes?", "Can you send it for 10 to 15 minutes?"]
references = ["Can it be delivered between 10 to 15 minutes?",
              "Can it be delivered between 10 to 15 minutes?"]


## Yours to write: `comet_inputs`

COMET scores a list of records, one per sentence: `{"src": ..., "mt": ..., "ref": ...}`, where
`mt` is the translation being scored. Without a reference, a record has only `src` and `mt`.
The function must also refuse lists of different lengths, because misaligned lines are scored
without complaint and give a number that means nothing. A11 calls this same function on your
A6 sentences.


In [ ]:
def comet_inputs(sources, translations, references=None):
    """Return one COMET record per sentence: src, mt, and ref when references are given.

    TODO: write it. Raise ValueError if translations, or references when given, are not the
    same length as sources.
    """
    raise NotImplementedError("write comet_inputs, then run this cell again")


# Cases whose answers you know.
assert comet_inputs(["hola"], ["hello"], ["hi"]) == [{"src": "hola", "mt": "hello", "ref": "hi"}], (
    "with a reference, each record is {'src': ..., 'mt': ..., 'ref': ...}")
assert comet_inputs(["hola"], ["hello"]) == [{"src": "hola", "mt": "hello"}], (
    "without a reference, each record has only 'src' and 'mt'")
for bad in ((["a", "b"], ["x"], None), (["a"], ["x"], ["r", "s"])):
    try:
        comet_inputs(*bad)
    except ValueError:
        continue
    raise AssertionError(f"comet_inputs must refuse lists of different lengths: {bad}")
print("comet_inputs passes its checks.")


## Predict, then score with a reference

Read the two systems' translations against the references. **Write a prediction** in
`PREDICTION`: which system will each model prefer, the one with a reference and the one without,
and why? Then the default COMET model scores both systems against the references. It needs no
account.


In [ ]:
PREDICTION = ""
if not PREDICTION.strip():
    print("No prediction written. Write one before scoring; the summary will say it is missing.\n")

import torch
from comet import download_model, load_from_checkpoint

GPUS = 1 if torch.cuda.is_available() else 0
systems = {"system 1": system_1, "system 2": system_2}


def score(model_name, with_references):
    """Each system's COMET system score from model_name, printing the sentence scores."""
    model = load_from_checkpoint(download_model(model_name))
    result = {}
    for name, translations in systems.items():
        records = comet_inputs(sources, translations, references if with_references else None)
        out = model.predict(records, batch_size=8, gpus=GPUS, num_workers=2, progress_bar=False)
        result[name] = out.system_score
        print(f"{name}: sentences {[round(s, 3) for s in out.scores]}, system {out.system_score:.3f}")
    return result


with_reference = score("Unbabel/wmt22-comet-da", with_references=True)


## Without a reference: COMETKiwi, which needs a Hugging Face token

COMETKiwi scores a translation from the source alone, and downloading it needs a free Hugging
Face account and a token.

1. Create a **Read** token at https://huggingface.co/settings/tokens, and copy it.
2. In Colab, click the key icon in the left sidebar, add a secret named `HF_TOKEN` with the token
   as its value, and turn on **Notebook access**.

Then run the next cell to log in.


In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")   # the secret's name, if you used a different one
if not HF_TOKEN:
    raise RuntimeError(
        "No HF_TOKEN found in Colab Secrets, so COMETKiwi cannot download. Follow the two steps "
        "in the cell above, turn on 'Notebook access' for this notebook, and run this cell again."
    )
login(token=HF_TOKEN)
print("Logged in to Hugging Face.")


In [ ]:
without_reference = score("Unbabel/wmt22-cometkiwi-da", with_references=False)


## What to record

The output of the cell below is what A10 asks you to turn in.


In [ ]:
def preferred(scores):
    return max(scores, key=scores.get)


print("=== For A10 ===")
for label, scores in (("With a reference (wmt22-comet-da)", with_reference),
                      ("Without one (wmt22-cometkiwi-da)", without_reference)):
    shown = ", ".join(f"{name} {value:.3f}" for name, value in scores.items())
    print(f"{label}: {shown};  prefers {preferred(scores)}")
same = preferred(with_reference) == preferred(without_reference)
print(f"The two models {'agree' if same else 'disagree'} on which system is better.")
print(f"Your prediction: {PREDICTION.strip() or '(none written)'}")
